# 第 4 章（Iterative Loop 版）

书里生成和批判用的是**同一个** `llm`。下面拆成两个，各用各的后端——这是本章最值得动手改的一处：
循环能不能收敛、以及收得快不快，取决于**批判者的判断力**，而生成者只要够快就行。

```python
PRODUCER = "local:qwen3.5-9b"           # 弱生成者：本地、无网络往返
CRITIC   = "deepseek:pro"               # 强批判者：准，但要等网络
```

kernel 选 `Python (agentic-book)`。本地那半需要 Ollama 在跑。
每轮会打上**耗时**，方便判断"弱生成 + 强批判"在**实时性**上是否真划算。

> 已知现象：弱批判者（本地 9B 当批判者）到第 3 轮会退化成重复句，被 2048 token 截断。
> 换成强批判者后这条消失——所以下面默认让强模型只干批判这一件事。

**本次运行的结果**：v1 就 10/10 满分，然后 v2–v10 **全崩**。原因不是批判者不行，而是
第 1 轮批评指出"doctest 里的 `<very large integer>` 会失败"，生成者就**照字面补上了
5000! 的真实数值**——16386 位数字，2048 token 全被数字吃光，代码在数字中间被截断。
之后 8 轮它每轮重蹈覆辙；强批判者每轮都准确诊断（"'函数体缺失'、'三引号未闭合'）、
第 2 轮甚至点出了根因，但它**不会动手改**。

10 轮共 792s，交付的是无法运行的代码——而 v1 只花了 54s。这说明：
**循环能否收敛取决于生成者的执行能力，不只是批判者的判断力**；
而且"多轮迭代"本身是实时性的敌人（上下文越长，本地推理越慢）。

In [1]:
import sys
import time
from pathlib import Path

# 【适配】找到含 common/ 的根目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.messages import SystemMessage, HumanMessage
from common import make_llm

# --- Configuration ---
# 【适配】原书是 load_dotenv() + 检查 OPENAI_API_KEY + 一个 ChatOpenAI("gpt-4o") 两用。
# 这里拆成两个模型：PRODUCER 负责写，CRITIC 负责挑，可以走不同后端。
PRODUCER = "local:qwen3.5-9b"     # 弱生成者（本地，无网络往返）
CRITIC   = "deepseek:pro"         # 强批判者（deepseek-v4-pro）


def build(spec: str):
    """'local:<模型名>' 或 'deepseek:flash|pro' → 一个 chat model。"""
    backend, _, name = spec.partition(":")
    if backend == "local":
        from ch04_reflection.local_llm import make_local_llm, server_is_up
        if not server_is_up():
            raise RuntimeError("Ollama 没在跑：~/.local/ollama/bin/ollama serve &")
        # 【适配】num_ctx 拉到 16384：10 轮下来 history 里堆了 10 份代码 + 10 份批评，
        # 8192 会在中途被截断，表现是"模型突然忘了任务要求"。
        return make_local_llm(model=name or "qwen3.5-9b:latest", temperature=0.1,
                              num_predict=2048, num_ctx=16384)
    if backend == "deepseek":
        return make_llm(strong=(name == "pro"), temperature=0.1)
    raise ValueError(f"未知后端: {spec}")


producer = build(PRODUCER)
critic = build(CRITIC)
print("生成者:", PRODUCER, "| 批判者:", CRITIC)


def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflection loop to progressively improve a Python function.
    """

    # --- The Core Task ---
    task_prompt = """
    Your task is to create a Python function named `calculate_factorial`.
    This function should do the following:
    1.  Accept a single integer `n` as input.
    2.  Calculate its factorial (n!).
    3.  Include a clear docstring explaining what the function does.
    4.  Handle edge cases: The factorial of 0 is 1.
    5.  Handle invalid input: Raise a ValueError if the input is a negative number.
    6.  The function must also work for large inputs such as n = 5000.
    """

    # --- The Reflection Loop ---
    max_iterations = 10
    current_code = ""
    message_history = [HumanMessage(content=task_prompt)]
    t_start = time.time()

    for i in range(max_iterations):
        print("\n" + "="*25 + f" REFLECTION LOOP: ITERATION {i + 1} " + "="*25)

        # --- 1. GENERATE / REFINE STAGE ---   ← 走 producer
        t0 = time.time()
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            response = producer.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = producer.invoke(message_history)
            current_code = response.content
        t_gen = time.time() - t0

        print("\n--- Generated Code (v" + str(i + 1) + ") ---\n" + current_code)
        print(f"\n[耗时] 生成 {t_gen:.1f}s")
        message_history.append(response)

        # --- 2. REFLECT STAGE ---   ← 走 critic
        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        reflector_prompt = [
            SystemMessage(content="""
                You are a senior software engineer and an expert in Python.
                Your role is to perform a meticulous code review.
                Critically evaluate the provided Python code based on the original task requirements.
                Look for bugs, style issues, missing edge cases, and areas for improvement.
                If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
                Otherwise, provide a bulleted list of your critiques.
            """),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        t0 = time.time()
        critique_response = critic.invoke(reflector_prompt)
        critique = critique_response.content
        t_crit = time.time() - t0
        print(f"\n[耗时] 批判 {t_crit:.1f}s（本轮合计 {t_gen + t_crit:.1f}s）")

        # --- 3. STOPPING CONDITION ---
        # 注：这里保持书里的子串匹配。它有个小坑——模型说
        #     "the code is NOT CODE_IS_PERFECT" 也会命中。见 README 的坑清单。
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break

        print("\n--- Critique ---\n" + critique)
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}"))

    print("\n" + "="*30 + " FINAL RESULT " + "="*30)
    print("\nFinal refined code after the reflection process:\n")
    print(current_code)
    print(f"\n[总计] {i + 1} 轮，墙钟耗时 {time.time() - t_start:.1f}s")


run_reflection_loop()

生成者: local:qwen3.5-9b | 批判者: deepseek:pro

========================= REFLECTION LOOP: ITERATION 1 =========================

>>> STAGE 1: GENERATING initial code...



--- Generated Code (v1) ---
```python
def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)  # Works for large inputs
        <very large integer>
    """
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")

    # Python's math.factorial is optimized and handles large integers efficiently.
    # For educational purposes, an iterative implementation is also shown below.
    # Using math.factorial for production due to its efficiency and robu


[耗时] 批判 38.3s（本轮合计 53.6s）

--- Critique ---
- `import math` is inside the function, which is non-idiomatic; it should be moved to the top of the module.
- The comment says an iterative implementation is “also shown below,” but no iterative implementation is actually included. This is misleading and should be updated or removed.
- The function delegates entirely to `math.factorial`. This is functionally correct and efficient, but if the goal is to implement factorial logic rather than wrap the standard library, the actual algorithm is missing.
- There is no explicit type check for `n`. `math.factorial` may accept or reject non-`int` inputs in ways that do not fully align with the stated contract of accepting a single integer.
- The doctest-style example for `calculate_factorial(5000)` uses a placeholder output (`<very large integer>`), which would not pass if doctests are executed.

========================= REFLECTION LOOP: ITERATION 2 =========================

>>> STAGE 1: REFINING 


--- Generated Code (v2) ---
```python
import math
from typing import Union

def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        TypeError: If n is not an integer.
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)
        127263569920529638470866290920845040402697345528000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


[耗时] 批判 40.0s（本轮合计 113.2s）

--- Critique ---
- The implementation is functionally correct for ordinary integers and uses `math.factorial`, so large inputs like `n=5000` are handled efficiently.
- The import `from typing import Union` is unused and should be removed.
- The doctest example for `calculate_factorial(5000)` contains an enormous, apparently truncated/inaccurate integer literal. If included as written, it will not match the actual output and will cause doctests to fail; it also makes the docstring unreadable.
- `isinstance(n, int)` returns `True` for booleans because `bool` is a subclass of `int`, so `calculate_factorial(True)` returns `1`. If booleans should be rejected, use a stricter type check such as `type(n) is int`.
- The function raises `TypeError` for non-integer inputs, which is reasonable but was not explicitly required; it should be intentional and consistently documented.

========================= REFLECTION LOOP: ITERATION 3 =========================

>>> STAG


--- Generated Code (v3) ---
```python
import math

def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        TypeError: If n is not an exact integer (e.g., float, str).
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)
        127263569920529638470866290920845040402697345528000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

... (11 outputs left out)


--- Generated Code (v9) ---
```python
import math

def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By definition, 0! = 1.

    Parameters:
        n (int): A non-negative integer whose factorial is to be computed.

    Returns:
        int: The factorial of n.

    Raises:
        TypeError: If n is not an exact integer (e.g., float, str, or boolean).
        ValueError: If n is negative.

    Examples:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(5000)
        12

[耗时] 生成 13.4s

>>> STAGE 2: REFLECTING on the generated code...



[耗时] 批判 15.2s（本轮合计 28.6s）

--- Critique ---
- The function has no implementation; its body consists only of the docstring, so every call returns `None` instead of the factorial.
- Negative inputs are not handled: `calculate_factorial(-1)` returns `None` instead of raising `ValueError`.
- `import math` is unused; the likely intended `return math.factorial(n)` is missing.
- The docstring examples are misleading: the `calculate_factorial(5000)` example shows `12`, which is not the factorial of 5000.
- The docstring promises a `TypeError` for non-integer inputs, but no type validation is implemented.
- Large inputs are not supported because no factorial calculation is performed at all.

========================= REFLECTION LOOP: ITERATION 10 =========================

>>> STAGE 1: REFINING code based on previous critique...



--- Generated Code (v10) ---
Here is the refined code. It now includes the actual implementation logic,

[耗时] 生成 6.4s

>>> STAGE 2: REFLECTING on the generated code...



[耗时] 批判 5.7s（本轮合计 12.1s）

--- Critique ---
- No actual code was provided after “Code to Review”; the implementation is missing.
- Unable to verify the function name, signature, docstring, factorial calculation, edge cases, or error handling.
- Please include the full Python code block for a meaningful review.

============================== FINAL RESULT ==============================

Final refined code after the reflection process:

Here is the refined code. It now includes the actual implementation logic,

[总计] 10 轮，墙钟耗时 792.0s
